# End-to-end pipeline (Stage 8)

Run the whole PAB pipeline — **ingest → discover → match → fit → figure →
report** — on a tiny **synthetic, offline** fixture (`pab.pipeline`). The
network/heavy steps are injected seams (`searcher`, `opener`), so this runs with
no network; the fit stage needs BING (skipped here if absent). Each stage is
idempotent, so a re-run resumes.

## 1. A synthetic offline fixture

Two profiles (with precomputed mixed-layer summaries), a discovery seam returning one granule each, and an `opener` returning a granule centred on each float.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, xarray as xr, pandas as pd
from pab import pipeline
from pab.db import Store
from pab.fit.models import FitConfig

def granule(center):
    nx=ny=5; clat,clon=center
    lat=np.linspace(clat-0.02,clat+0.02,nx); lon=np.linspace(clon-0.02,clon+0.02,ny)
    lon2d,lat2d=np.meshgrid(lon,lat); wave=np.linspace(400,700,31)
    rrs=np.tile(0.01*np.exp(-(wave-400)/150.),(nx,ny,1))
    return xr.Dataset({"Rrs":(("x","y","wl"),rrs),"Rrs_unc":(("x","y","wl"),rrs*0.05),
                       "l2_flags":(("x","y"),np.zeros((nx,ny),int))},
                      coords={"latitude":(("x","y"),lat2d),"longitude":(("x","y"),lon2d),
                              "wavelength":("wl",wave)})

profiles=[
  {"wmo":7902226,"cycle":5,"latitude":20.,"longitude":-50.,"time":"2025-02-18T20:00:00",
   "summary":{"mld":30.,"mld_method":"x","bbp700":1e-3,"chla":0.1,"n_points":6}},
  {"wmo":7902136,"cycle":8,"latitude":4.,"longitude":-137.,"time":"2025-01-25T13:00:00",
   "summary":{"mld":30.,"mld_method":"x","bbp700":2e-3,"chla":0.2,"n_points":6}}]
gid=lambda la,lo:f"PACE_{la:.0f}_{lo:.0f}"
def searcher(lat,lon,t0,t1,cfg):
    mid=t0+(t1-t0)/2; g=gid(lat,lon)
    return pd.DataFrame([{"id":g,"time":mid.isoformat(),"polygon":None,"CC":5.,"url":f"s3://b/{g}.nc"}])
grans={f"s3://b/{gid(p['latitude'],p['longitude'])}.nc":granule((p['latitude'],p['longitude'])) for p in profiles}
opener=lambda src:grans[src]
print("fixture:", len(profiles), "profiles")

fixture: 2 profiles


## 2. Run the pipeline (dry run, then for real)

In [2]:
import tempfile
cfg = pipeline.PipelineConfig(profiles=profiles, outdir=tempfile.mkdtemp(),
                              fit=FitConfig(nsteps=400, nburn=100, analysis_burn=100),
                              make_figures=False)
print("plan:", pipeline.run(None, cfg, dry_run=True)["stages"])

store = Store.open(":memory:")
summary = pipeline.run(store, cfg, opener=opener, searcher=searcher)
for stage, res in summary.items():
    print(f"{stage:9s}: {res if stage!='report' else {k:res[k] for k in ('n_uploaded','manifest')}}")

plan: ['ingest', 'discover', 'match', 'fit', 'figure', 'report']


ingest   : {'written': ['7902226_5', '7902136_8'], 'skipped': []}
discover : {'granules_upserted': 2}
match    : {'written': ['7902226_5_PACE_20_-50', '7902136_8_PACE_4_-137'], 'skipped': [], 'unmatched': []}
fit      : {'written': [], 'skipped': [], 'failed': ['7902136_8_PACE_4_-137_2_2_ExpBPow', '7902226_5_PACE_20_-50_2_2_ExpBPow']}
figure   : {'written': [], 'skipped': [], 'failed': []}
report   : {'n_uploaded': 0, 'manifest': '/tmp/tmpwes7j0y4/release/manifest.json'}


## 3. What landed in the store + on disk

In [3]:
for t in ("floats","profiles","mld_summary","granules","matchups","matchup_pixels","fits","fit_results"):
    print(f"  {t:15s} {store.count(t)}")
import os
print("\nsite pages:", sorted(os.listdir(cfg.out()/"site")))
print("release:", sorted(os.listdir(cfg.out()/"release")))

  floats          2
  profiles        2
  mld_summary     2
  granules        2
  matchups        2
  matchup_pixels  20
  fits            0
  fit_results     0

site pages: ['aggregates.rst', 'conf.py', 'index.rst', 'methods.rst', 'summary.rst']
release: ['manifest.json', 'matchup_summary.csv', 'matchup_summary.parquet', 'store']


## 4. Idempotent re-run (resumes — writes nothing new)

In [4]:
again = pipeline.run(store, cfg, stages=("match","fit"), opener=opener)
print("match written:", again["match"]["written"], "| skipped:", len(again["match"]["skipped"]))
print("fit   written:", again["fit"]["written"], "| skipped:", len(again["fit"]["skipped"]))

match written: [] | skipped: 2
fit   written: [] | skipped: 0


## 5. (Optional) live run

With network + an Earthdata Login, drop the `opener`/`searcher` seams and the
inline `summary` (so `ingest` fetches via argopy and `discover` queries
earthaccess), or just use the CLI: `pab --db pab.db` (or `pab --dry-run`).

In [5]:
RUN_LIVE = False
if RUN_LIVE:
    live = pipeline.PipelineConfig()   # dev_profiles.csv, gdac/expert, live discovery
    with Store.open("pab.db") as s:
        print(pipeline.run(s, live))